# 02 Train: one holdout, six candidates, `@champion` / `@challenger`

- **Target:** units sold on complete store-SKU-days that never stocked out. On those days sales
  equal demand; on stocked-out days sales are censored, so they are never used as a target.
- **Training set:** labels from `gold.store_sku_daily`, features through `FeatureLookup` with a
  point-in-time join on `ml.store_sku_features` (timeseries key `date`).
- **Windows** (derived from the generator calendar): training from the 29th day of history to
  the day before the holdout; Optuna tunes the two boosters on a 5-week validation window just
  before the holdout; **one holdout** for every candidate: the last 8 weeks of history.
- **Metric:** WAPE (sum |actual - forecast| / sum actual) on uncensored holdout days, plus bias.
- **Candidates:** the incumbent 4-week moving average, seasonal naive (same weekday last week),
  Ridge, HistGradientBoosting, XGBoost and LightGBM (the last two tuned with Optuna).
- **Registry:** the best two fitted models become versions of `ml.demand_lgbm` (the name is
  fixed whichever algorithm wins) with aliases `@champion` and `@challenger`.

In [ ]:
import datetime as dt
import importlib.metadata as md
import json
import os
import sys
import time

import mlflow
import numpy as np
import optuna
import pandas as pd
import yaml
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F

sys.path.append(os.path.abspath("../generator"))
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup  # noqa: E402

from bigbox_ml.features import CATEGORICAL, FEATURES, NUMERIC  # noqa: E402

dbutils.widgets.text("catalog", "bigbox_fe_bar")
dbutils.widgets.text("experiment", "/Users/me/bigbox_demand_forecast")
dbutils.widgets.text("sample_fraction", "0.35")
dbutils.widgets.text("trials", "12")
CAT = dbutils.widgets.get("catalog")
EXPERIMENT = dbutils.widgets.get("experiment")
SAMPLE = float(dbutils.widgets.get("sample_fraction"))
TRIALS = int(dbutils.widgets.get("trials"))
FT = f"{CAT}.ml.store_sku_features"
MODEL = f"{CAT}.ml.demand_lgbm"
SEED = 20261023

with open("../generator/config.yaml") as f:
    cal = yaml.safe_load(f)["calendar"]
HOURS = cal["close_hour"] - cal["open_hour"]
TRAIN_START = cal["history_start"] + dt.timedelta(days=28)
HOLDOUT_END = cal["history_end"]
HOLDOUT_START = HOLDOUT_END - dt.timedelta(days=55)
VAL_START = HOLDOUT_START - dt.timedelta(days=35)
print(
    f"train {TRAIN_START} .. {HOLDOUT_START - dt.timedelta(days=1)} "
    f"(Optuna: fit to {VAL_START - dt.timedelta(days=1)}, validate {VAL_START} .. "
    f"{HOLDOUT_START - dt.timedelta(days=1)}); holdout {HOLDOUT_START} .. {HOLDOUT_END}"
)

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT)
optuna.logging.set_verbosity(optuna.logging.WARNING)
fe = FeatureEngineeringClient()

train 2025-10-29 .. 2026-08-05 (Optuna: fit to 2026-07-01, validate 2026-07-02 .. 2026-08-05); holdout 2026-08-06 .. 2026-09-30


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


## Labels and point-in-time training set

In [ ]:
daily = spark.read.table(f"{CAT}.gold.store_sku_daily")
uncensored = daily.where((F.col("stockout_hours") == 0) & (F.col("trading_hours") == HOURS))
labels = uncensored.select(
    "store_id", "sku_id", "date", F.col("units_sold").cast("double").alias("label")
)
train_labels = labels.where(
    F.col("date").between(F.lit(TRAIN_START), F.lit(HOLDOUT_START - dt.timedelta(days=1)))
).sample(fraction=SAMPLE, seed=SEED)
holdout_labels = labels.where(F.col("date").between(F.lit(HOLDOUT_START), F.lit(HOLDOUT_END)))
lookup = [
    FeatureLookup(
        table_name=FT,
        lookup_key=["store_id", "sku_id"],
        timestamp_lookup_key="date",
        feature_names=FEATURES,
    )
]


def load(lbl):
    ts = fe.create_training_set(df=lbl, feature_lookups=lookup, label="label")
    df = ts.load_df().toPandas()
    df["date"] = pd.to_datetime(df["date"]).dt.date  # the point-in-time join returns a timestamp
    return ts, df


t0 = time.time()
train_set, train = load(train_labels)
_, holdout = load(holdout_labels)
for c in CATEGORICAL:
    train[c], holdout[c] = train[c].astype(str), holdout[c].astype(str)
print(
    f"train rows {len(train):,} (sample {SAMPLE:.0%} of uncensored days), holdout rows "
    f"{len(holdout):,} (all uncensored days), loaded in {time.time() - t0:.0f} s"
)
censored = (
    daily.where(F.col("date").between(F.lit(HOLDOUT_START), F.lit(HOLDOUT_END)))
    .agg(
        F.count("*").alias("all"),
        F.sum((F.col("stockout_hours") > 0).cast("int")).alias("censored"),
    )
    .first()
)
print(
    f"holdout store-SKU-days {censored['all']:,}, "
    f"of which censored (excluded) {censored['censored']:,}"
)

train rows 2,527,219 (sample 35% of uncensored days), holdout rows 1,425,660 (all uncensored days), loaded in 106 s
holdout store-SKU-days 1,568,280, of which censored (excluded) 142,620


In [ ]:
def wape(y, p):
    return float(np.abs(y - p).sum() / y.sum())


def bias(y, p):
    return float((p.sum() - y.sum()) / y.sum())


y_tr, y_ho = train["label"].to_numpy(), holdout["label"].to_numpy()
TUNE_ROWS = 800_000  # Optuna fits on a fixed sample so 2 x TRIALS fits stay within the budget
tune_fit = train[train["date"] < VAL_START]
tune_fit = tune_fit.sample(n=min(TUNE_ROWS, len(tune_fit)), random_state=SEED)
tune_val = train[train["date"] >= VAL_START]
print(f"tuning: fit {len(tune_fit):,} rows, validate {len(tune_val):,} rows")

tuning: fit 800,000 rows, validate 318,369 rows


## Candidates
Every fitted candidate is a scikit-learn `Pipeline` (encoding + estimator), so the registered
artifact takes the raw feature row and nothing is re-implemented at serving time.

In [ ]:
from lightgbm import LGBMRegressor, early_stopping  # noqa: E402
from sklearn.compose import ColumnTransformer  # noqa: E402
from sklearn.ensemble import HistGradientBoostingRegressor  # noqa: E402
from sklearn.impute import SimpleImputer  # noqa: E402
from sklearn.linear_model import Ridge  # noqa: E402
from sklearn.pipeline import Pipeline  # noqa: E402
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler  # noqa: E402
from xgboost import XGBRegressor  # noqa: E402

N_JOBS = os.cpu_count()


def ordinal():
    return ColumnTransformer(
        [
            (
                "cat",
                OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                CATEGORICAL,
            )
        ],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )


def onehot():
    num = [c for c in NUMERIC if c != "dow"]
    return ColumnTransformer(
        [
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL + ["dow"]),
            (
                "num",
                Pipeline(
                    [("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]
                ),
                num,
            ),
        ]
    )


enc = ordinal().fit(tune_fit[FEATURES])
Xf, Xv = enc.transform(tune_fit[FEATURES]), enc.transform(tune_val[FEATURES])
yf, yv = tune_fit["label"].to_numpy(), tune_val["label"].to_numpy()


def tune(name, make, fit, space):
    """Optuna on the validation window; every trial is a nested MLflow run."""

    def objective(trial):
        params = space(trial)
        with mlflow.start_run(run_name=f"{name}-trial-{trial.number}", nested=True):
            m = make(params)
            best_iter = fit(m, trial)
            score = wape(yv, m.predict(Xv))
            mlflow.log_params(params)
            mlflow.log_metrics({"val_wape": score, "best_iteration": best_iter})
        trial.set_user_attr("best_iteration", best_iter)
        return score

    with mlflow.start_run(run_name=f"{name}-optuna", nested=True):
        study = optuna.create_study(
            direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED)
        )
        study.optimize(objective, n_trials=TRIALS)
        mlflow.log_metric("best_val_wape", study.best_value)
    best = dict(study.best_params, n_estimators=study.best_trial.user_attrs["best_iteration"])
    return best, study.best_value


def lgbm_space(t):
    return {
        "num_leaves": t.suggest_int("num_leaves", 31, 255, log=True),
        "learning_rate": t.suggest_float("learning_rate", 0.03, 0.2, log=True),
        "min_child_samples": t.suggest_int("min_child_samples", 20, 400, log=True),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }


def lgbm(params):
    return LGBMRegressor(
        objective="poisson",
        subsample_freq=1,
        n_estimators=params.pop("n_estimators", 2000),
        random_state=SEED,
        n_jobs=N_JOBS,
        verbose=-1,
        **params,
    )


def lgbm_fit(m, trial):
    m.fit(Xf, yf, eval_set=[(Xv, yv)], callbacks=[early_stopping(50, verbose=False)])
    return int(m.best_iteration_)


def xgb_space(t):
    return {
        "max_depth": t.suggest_int("max_depth", 4, 10),
        "learning_rate": t.suggest_float("learning_rate", 0.03, 0.2, log=True),
        "min_child_weight": t.suggest_float("min_child_weight", 1, 50, log=True),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }


def xgb(params, early=True):
    n = params.pop("n_estimators", 2000)
    return XGBRegressor(
        objective="count:poisson",
        tree_method="hist",
        n_estimators=n,
        random_state=SEED,
        n_jobs=N_JOBS,
        early_stopping_rounds=50 if early else None,
        **params,
    )


def xgb_fit(m, trial):
    m.fit(Xf, yf, eval_set=[(Xv, yv)], verbose=False)
    return int(m.best_iteration) + 1

## Sweep: tune, fit on the full training window, score the shared holdout

In [ ]:
SERVING_PACKAGES = (
    "mlflow",
    "scikit-learn",
    "lightgbm",
    "xgboost",
    "pandas",
    "numpy",
    "cloudpickle",
)


def installed_version(package: str) -> str | None:
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return None


installed = {p: v for p in SERVING_PACKAGES if (v := installed_version(p))}
reqs = [f"{p}=={v}" for p, v in installed.items()]  # pinned to this runtime, so serving matches
print("model pip requirements:", reqs)
X_tr, X_ho = train[FEATURES], holdout[FEATURES]
signature = infer_signature(X_tr.head(100), np.zeros(100))
results = []

with mlflow.start_run(run_name=f"sweep-{dt.datetime.now(dt.UTC):%Y%m%dT%H%M%S}") as sweep:
    mlflow.log_params(
        {
            "train_start": str(TRAIN_START),
            "holdout_start": str(HOLDOUT_START),
            "holdout_end": str(HOLDOUT_END),
            "sample_fraction": SAMPLE,
            "trials": TRIALS,
        }
    )
    mlflow.log_input(mlflow.data.load_delta(table_name=FT), context="features")
    mlflow.log_input(
        mlflow.data.load_delta(table_name=f"{CAT}.gold.store_sku_daily"), context="labels"
    )

    def record(
        name, algorithm, pred, run_id, tuned=False, val=None, fit_s=0.0, params=None, rows=0
    ):
        r = {
            "candidate": name,
            "algorithm": algorithm,
            "run_id": run_id,
            "tuned": tuned,
            "val_wape": val,
            "holdout_wape": wape(y_ho, pred),
            "holdout_bias": bias(y_ho, pred),
            "holdout_rows": len(y_ho),
            "train_rows": rows,
            "fit_seconds": round(fit_s, 1),
            "params": json.dumps(params or {}, sort_keys=True),
        }
        results.append(r)
        print(
            f"{name:<22} holdout WAPE {r['holdout_wape']:.4f}  "
            f"bias {r['holdout_bias']:+.4f}  ({fit_s:.0f} s)"
        )
        return r

    for name, algorithm, pred in [
        ("incumbent_ma28", "28-day moving average of sales", X_ho["mean_28"].fillna(0).to_numpy()),
        (
            "seasonal_naive",
            "same weekday last week",
            X_ho["lag_7"].fillna(X_ho["mean_28"]).fillna(0).to_numpy(),
        ),
    ]:
        with mlflow.start_run(run_name=name, nested=True) as run:
            r = record(name, algorithm, pred, run.info.run_id)
            mlflow.log_metrics(
                {"holdout_wape": r["holdout_wape"], "holdout_bias": r["holdout_bias"]}
            )

    lgbm_best, lgbm_val = tune("lightgbm", lgbm, lgbm_fit, lgbm_space)
    xgb_best, xgb_val = tune("xgboost", lambda p: xgb(p), xgb_fit, xgb_space)
    fitted = [
        (
            "ridge",
            "Ridge (one-hot + scaled)",
            lambda: Pipeline([("prep", onehot()), ("model", Ridge(alpha=1.0))]),
            {"alpha": 1.0},
            False,
            None,
        ),
        (
            "hist_gradient_boosting",
            "HistGradientBoosting, Poisson loss",
            lambda: Pipeline(
                [
                    ("prep", ordinal()),
                    (
                        "model",
                        HistGradientBoostingRegressor(
                            loss="poisson",
                            max_iter=500,
                            learning_rate=0.1,
                            categorical_features=[0, 1],
                            early_stopping=True,
                            random_state=SEED,
                        ),
                    ),
                ]
            ),
            {"max_iter": 500, "learning_rate": 0.1},
            False,
            None,
        ),
        (
            "xgboost",
            "XGBoost, Poisson, Optuna",
            lambda: Pipeline([("prep", ordinal()), ("model", xgb(dict(xgb_best), early=False))]),
            xgb_best,
            True,
            xgb_val,
        ),
        (
            "lightgbm",
            "LightGBM, Poisson, Optuna",
            lambda: Pipeline([("prep", ordinal()), ("model", lgbm(dict(lgbm_best)))]),
            lgbm_best,
            True,
            lgbm_val,
        ),
    ]
    for name, algorithm, make, params, tuned, val in fitted:
        with mlflow.start_run(run_name=name, nested=True) as run:
            t0 = time.time()
            pipe = make().fit(X_tr, y_tr)
            fit_s = time.time() - t0
            pred = pipe.predict(X_ho)
            r = record(name, algorithm, pred, run.info.run_id, tuned, val, fit_s, params, len(y_tr))
            mlflow.log_params({k: v for k, v in params.items()})
            mlflow.log_metrics(
                {k: r[k] for k in ("holdout_wape", "holdout_bias", "fit_seconds")}
                | ({"val_wape": val} if val is not None else {})
            )
            mlflow.sklearn.log_model(
                pipe,
                name="model",
                signature=signature,
                input_example=X_tr.head(5),
                pip_requirements=reqs,
            )

model pip requirements: ['mlflow==3.18.0', 'scikit-learn==1.6.1', 'lightgbm==4.7.0', 'xgboost==3.4.1', 'pandas==2.2.3', 'numpy==2.1.3', 'cloudpickle==3.0.0']


2026/11/24 07:33:58 WARNING mlflow.data.spark_delta_utils: Failed to obtain version information for Delta table with name 'bigbox_fe_bar.ml.store_sku_features'. Version information may not be included in the dataset source for MLflow Tracking. Exception: [JVM_ATTRIBUTE_NOT_SUPPORTED] Attribute `_jvm` is not supported in Spark Connect as it depends on the JVM. If you need to use this attribute, do not use Spark Connect when creating your session. Visit https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession for creating regular Spark Session in detail.
2026/11/24 07:33:59 WARNING mlflow.data.spark_dataset: Encountered an unexpected exception while computing Spark dataset profile. Exception: [NOT_IMPLEMENTED] rdd is not implemented.
2026/11/24 07:33:59 WARNING mlflow.data.spark_delta_utils: Failed to obtain version information for Delta table with name 'bigbox_fe_bar.gold.store_sku_daily'. Version information may not be included in the dataset source for

incumbent_ma28         holdout WAPE 0.4790  bias -0.0054  (0 s)
seasonal_naive         holdout WAPE 0.6208  bias -0.0099  (0 s)


/local_disk0/.ephemeral_nfs/envs/pythonEnv-232b3b6f-887d-4566-86af-5458bd6f90d8/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-232b3b6f-887d-4566-86af-5458bd6f90d8/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-232b3b6f-887d-4566-86af-5458bd6f90d8/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
/local_disk0/.ephemeral_nfs/envs/pythonEnv-232b3b

ridge                  holdout WAPE 0.4712  bias +0.0045  (18 s)


🔗 View Logged Model at: https://fevm-serverless-stable-ritesh.cloud.databricks.com/ml/experiments/3164772271447205/models/m-d43fd73cbf47442ea92b1f65b9034511?o=7474657773731312
Unexpected status resolving file endpoints (403, request ID a85c192b-1991-4ed8-9ace-a73549c0685b), using the default endpoints.


... (2 outputs left out)

xgboost                holdout WAPE 0.4569  bias +0.0025  (105 s)


🔗 View Logged Model at: https://fevm-serverless-stable-ritesh.cloud.databricks.com/ml/experiments/3164772271447205/models/m-08c1f00f52d44bbc869ad242810c5bef?o=7474657773731312
Unexpected status resolving file endpoints (403, request ID b59a38b6-9877-4e17-82bf-0ec26791373c), using the default endpoints.


lightgbm               holdout WAPE 0.4562  bias +0.0025  (81 s)


🔗 View Logged Model at: https://fevm-serverless-stable-ritesh.cloud.databricks.com/ml/experiments/3164772271447205/models/m-f6438e7559aa4e0aa052066c95be0c66?o=7474657773731312
Unexpected status resolving file endpoints (403, request ID 63caf765-d3d7-4933-aeba-da40cf781907), using the default endpoints.


## AutoML forecasting (attempted once)

In [ ]:
try:
    from databricks import automl  # noqa: F401

    print("databricks.automl is importable:", automl.__file__)
except ImportError as e:
    print(f"AutoML forecasting not a candidate here: {type(e).__name__}: {e}")
    print(
        "Serverless environment 4 has no databricks.automl; AutoML forecasting would also fit one "
        "series per store x SKU (28,005 series) without the shared features above."
    )

AutoML forecasting not a candidate here: ImportError: cannot import name 'automl' from 'databricks' (/databricks/python_shell/lib/databricks/__init__.py)
Serverless environment 4 has no databricks.automl; AutoML forecasting would also fit one series per store x SKU (28,005 series) without the shared features above.


## Register the best two fitted models

In [ ]:
sweep_df = pd.DataFrame(results).sort_values("holdout_wape").reset_index(drop=True)
models = sweep_df[sweep_df["train_rows"] > 0].head(2)
client = MlflowClient(registry_uri="databricks-uc")
sweep_df["model_version"], sweep_df["alias"] = None, None
for alias, (_, row) in zip(("champion", "challenger"), models.iterrows(), strict=True):
    mv = mlflow.register_model(f"runs:/{row['run_id']}/model", MODEL)
    client.set_registered_model_alias(MODEL, alias, mv.version)
    client.update_model_version(
        MODEL,
        mv.version,
        description=(
            f"{row['algorithm']}. Holdout ({HOLDOUT_START}..{HOLDOUT_END}) "
            f"WAPE {row['holdout_wape']:.4f}, bias {row['holdout_bias']:+.4f} "
            f"on {row['holdout_rows']:,} uncensored store-SKU-days. "
            f"MLflow run {row['run_id']}."
        ),
    )
    for k in ("candidate", "algorithm"):
        client.set_model_version_tag(MODEL, mv.version, k, row[k])
    client.set_model_version_tag(MODEL, mv.version, "holdout_wape", f"{row['holdout_wape']:.4f}")
    sweep_df.loc[sweep_df["run_id"] == row["run_id"], ["model_version", "alias"]] = [
        mv.version,
        alias,
    ]
    print(f"@{alias}: version {mv.version} = {row['candidate']}")

champ = sweep_df[sweep_df["alias"] == "champion"].iloc[0]
table = "\n".join(
    f"| {r.candidate} | {r.holdout_wape:.4f} | {r.holdout_bias:+.4f} | {r.alias or ''} |"
    for r in sweep_df.itertuples()
)
client.update_registered_model(
    MODEL,
    description=f"""Daily demand forecast per ranged store x SKU for Bigbox Stores (synthetic data).

**Champion: {champ["algorithm"]}** (registered under this name whichever algorithm wins).

- Predicts: units demanded (in store + online requested) on day d, from features known
  at the start of d (`{FT}`).
- Trained on uncensored days only ({TRAIN_START} .. {HOLDOUT_START - dt.timedelta(days=1)},
  {SAMPLE:.0%} sample); stocked-out days are censored and never a target.
- Holdout: {HOLDOUT_START} .. {HOLDOUT_END}, every candidate on the same
  {champ["holdout_rows"]:,} uncensored store-SKU-days.

| candidate | holdout WAPE | bias | alias |
|---|---|---|---|
{table}

Limits: synthetic data from one seeded generator; a forecast for day d+k reuses the history
columns as of day d (only the plan columns move); not tested beyond the 8-week holdout.""",
)

Successfully registered model 'bigbox_fe_bar.ml.demand_lgbm'.
2026/11/24 08:07:53 WARNING mlflow.tracking._model_registry.fluent: Run with id e5ad919ec29742c9830a1a775823d2b7 has no artifacts at artifact path 'model', registering model based on models:/m-f6438e7559aa4e0aa052066c95be0c66 instead
Unexpected status resolving file endpoints (403, request ID edcc5581-f90b-4d64-9053-68ba9b96746a), using the default endpoints.


{'text/plain': 'Downloading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': '55fb56f8df5648008b255b4eb83a9d41'}}

{'text/plain': 'Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': '2bcf0fd8ea3d481191103f2ba217deb7'}}

Unexpected status resolving file endpoints (403, request ID 700e2ad5-025b-4f00-a5c8-b41f0edde1ec), using the default endpoints.
🔗 Created version '1' of model 'bigbox_fe_bar.ml.demand_lgbm': https://fevm-serverless-stable-ritesh.cloud.databricks.com/explore/data/models/bigbox_fe_bar/ml/demand_lgbm/version/1?o=7474657773731312


@champion: version 1 = lightgbm


Registered model 'bigbox_fe_bar.ml.demand_lgbm' already exists. Creating a new version of this model...
2026/11/24 08:08:01 WARNING mlflow.tracking._model_registry.fluent: Run with id b737fbe8096249c483e0e249ab84a868 has no artifacts at artifact path 'model', registering model based on models:/m-08c1f00f52d44bbc869ad242810c5bef instead
Unexpected status resolving file endpoints (403, request ID b8c03118-b40d-4e4b-b6d3-6806e8c9b786), using the default endpoints.


{'text/plain': 'Downloading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': '6b00962721144780b7ac03dbb22d97d6'}}

{'text/plain': 'Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': 'b65d606d5ff44b8aaaf62efb7e93fcb5'}}

Unexpected status resolving file endpoints (403, request ID d6b41d14-82b4-40fe-93f8-ddad743e0d4b), using the default endpoints.
🔗 Created version '2' of model 'bigbox_fe_bar.ml.demand_lgbm': https://fevm-serverless-stable-ritesh.cloud.databricks.com/explore/data/models/bigbox_fe_bar/ml/demand_lgbm/version/2?o=7474657773731312


@challenger: version 2 = xgboost


{'text/plain': "<RegisteredModel: aliases={}, creation_timestamp=1795507673392, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description=('Daily demand forecast per ranged store x SKU for Bigbox Stores (synthetic '\n 'data).\\n'\n '\\n'\n '**Champion: LightGBM, Poisson, Optuna** (registered under this name '\n 'whichever algorithm wins).\\n'\n '\\n'\n '- Predicts: units demanded (in store + online requested) on day d, from '\n 'features known\\n'\n '  at the start of d (`bigbox_fe_bar.ml.store_sku_features`).\\n'\n '- Trained on uncensored days only (2025-10-29 .. 2026-08-05,\\n'\n '  35% sample); stocked-out days are censored and never a target.\\n'\n '- Holdout: 2026-08-06 .. 2026-09-30, every candidate on the same\\n'\n '  1,425,660 uncensored store-SKU-days.\\n'\n '\\n'\n '| candidate | holdout WAPE | bias | alias |\\n'\n '|---|---|---|---|\\n'\n '| lightgbm | 0.4562 | +0.0025 | champion |\\n'\n '| xgboost | 0.4569 | +0.0025 | challenger

In [ ]:
spark.createDataFrame(sweep_df.assign(run_at=dt.datetime.now(dt.UTC))).createOrReplaceTempView(
    "sweep"
)
spark.sql(f"""INSERT OVERWRITE {CAT}.ml.sweep_results
  SELECT candidate, algorithm, run_id, tuned, nanvl(val_wape, CAST(NULL AS DOUBLE)),
         holdout_wape, holdout_bias,
         CAST(holdout_rows AS BIGINT), CAST(train_rows AS BIGINT), fit_seconds, params,
         model_version, alias, run_at FROM sweep""")
display(spark.read.table(f"{CAT}.ml.sweep_results").orderBy("holdout_wape"))

| candidate | algorithm | run_id | tuned | val_wape | holdout_wape | holdout_bias | holdout_rows | train_rows | fit_seconds | params | model_version | alias | run_at |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| lightgbm | LightGBM, Poisson, Optuna | e5ad919ec29742c9830a1a775823d2b7 | True | 0.4563967862741929 | 0.4561919240315781 | 0.002465622050676582 | 1425660 | 2527219 | 81.2 | {"colsample_bytree": 0.9257376184599424, "learning_rate": 0.08631445238621013, "min_child_samples": 396, "n_estimators": 137, "num_leaves": 183, "reg_lambda": 5.020265261533734, "subsample": 0.710756326146477} | 1 | champion | 2026-11-24T08:08:09.354Z |
| xgboost | XGBoost, Poisson, Optuna | b737fbe8096249c483e0e249ab84a868 | True | 0.4568869168544296 | 0.4568692622573542 | 0.0025297442586663505 | 1425660 | 2527219 | 104.7 | {"colsample_bytree": 0.9011641584515879, "learning_rate": 0.04242178951552486, "max_depth": 4, "min_child_weight": 10.935738780012137, "n_estimators": 790, "reg_lambda": 0.031849393679705316, "subsample": 0.9747509850672788} | 2 | challenger | 2026-11-24T08:08:09.354Z |
| hist_gradient_boosting | HistGradientBoosting, Poisson loss | 48913f281a7a44b496ef70be0fc64063 | False |  | 0.4570166800729174 | 0.0014017255190698092 | 1425660 | 2527219 | 73.8 | {"learning_rate": 0.1, "max_iter": 500} |  |  | 2026-11-24T08:08:09.354Z |
| ridge | Ridge (one-hot + scaled) | 28dafde5b01c4549b9162bf6d1ff8d2c | False |  | 0.4712485936933085 | 0.004488877722598982 | 1425660 | 2527219 | 17.9 | {"alpha": 1.0} |  |  | 2026-11-24T08:08:09.354Z |
| incumbent_ma28 | 28-day moving average of sales | a9c859ed25624013b5d055504aa695a8 | False |  | 0.47895656408493553 | -0.005389570678540163 | 1425660 | 0 | 0.0 | {} |  |  | 2026-11-24T08:08:09.354Z |
| seasonal_naive | same weekday last week | 7f9c989d96d24f4498958f19b455095b | False |  | 0.6208029694116014 | -0.00990563342769818 | 1425660 | 0 | 0.0 | {} |  |  | 2026-11-24T08:08:09.354Z |


## Champion vs incumbent by category (holdout)

In [ ]:
champ_model = mlflow.sklearn.load_model(f"models:/{MODEL}@champion")
h = holdout.assign(champion=champ_model.predict(X_ho), incumbent=X_ho["mean_28"].fillna(0))
by_cat = (
    h.groupby("category")
    .apply(
        lambda g: pd.Series(
            {
                "rows": len(g),
                "actual_units": g["label"].sum(),
                "wape_incumbent": wape(g["label"], g["incumbent"]),
                "wape_champion": wape(g["label"], g["champion"]),
            }
        ),
        include_groups=False,
    )
    .round(4)
)
print(by_cat.to_string())

{'text/plain': 'Downloading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]', 'application/vnd.jupyter.widget-view+json': {'version_major': 2, 'version_minor': 0, 'model_id': '66d70c132d0649599aba81fda43dfc2e'}}

Unexpected status resolving file endpoints (403, request ID aef9a22a-4538-4907-aa08-b91c48a8c8a7), using the default endpoints.


                    rows  actual_units  wape_incumbent  wape_champion
category                                                             
Apparel          64215.0      211417.0          0.4986         0.4800
Bakery           74857.0      431282.0          0.4435         0.4194
Beverages       152914.0      663612.0          0.4895         0.4644
Dairy           123487.0      561452.0          0.4752         0.4535
Electronics      51045.0       37224.0          0.8120         0.8119
Frozen          104967.0      231511.0          0.6042         0.5874
Home & Garden    85622.0       76851.0          0.8387         0.8234
Household       146350.0      330365.0          0.5898         0.5739
Meat & Seafood   88564.0      227265.0          0.5729         0.5509
Pantry          211742.0     1374783.0          0.4254         0.4022
Personal Care   117809.0      306735.0          0.5528         0.5354
Produce         151762.0     1296700.0          0.4125         0.3874
Toys             523

In [ ]:
dbutils.notebook.exit(
    json.dumps(
        {
            "model": MODEL,
            "champion": {
                "version": str(champ["model_version"]),
                "candidate": champ["candidate"],
                "holdout_wape": round(champ["holdout_wape"], 4),
            },
            "sweep": [
                {k: r[k] for k in ("candidate", "holdout_wape", "holdout_bias")} for r in results
            ],
            "train_rows": len(train),
            "holdout_rows": len(holdout),
            "experiment": EXPERIMENT,
            "sweep_run_id": sweep.info.run_id,
        },
        default=str,
    )
)

{"model": "bigbox_fe_bar.ml.demand_lgbm", "champion": {"version": "1", "candidate": "lightgbm", "holdout_wape": 0.4562}, "sweep": [{"candidate": "incumbent_ma28", "holdout_wape": 0.47895656408493553, "holdout_bias": -0.005389570678540163}, {"candidate": "seasonal_naive", "holdout_wape": 0.6208029694116014, "holdout_bias": -0.00990563342769818}, {"candidate": "ridge", "holdout_wape": 0.4712485936933085, "holdout_bias": 0.004488877722598982}, {"candidate": "hist_gradient_boosting", "holdout_wape": 0.4570166800729174, "holdout_bias": 0.0014017255190698092}, {"candidate": "xgboost", "holdout_wape": 0.4568692622573542, "holdout_bias": 0.0025297442586663505}, {"candidate": "lightgbm", "holdout_wape": 0.4561919240315781, "holdout_bias": 0.002465622050676582}], "train_rows": 2527219, "holdout_rows": 1425660, "experiment": "/Users/ritesh.patel@databricks.com/bigbox_demand_forecast", "sweep_run_id": "1a50b5dfe8ae4d5fa25fcbad139e9555"}